# 1. Business Understanding

$$\boxed{\textbf{\textit{Business Understanding}}} \longrightarrow \text{Data Understanding} \longrightarrow \text{Data Preparation} \longrightarrow \text{Modeling} \longrightarrow \text{Evaluation} \longrightarrow \text{Deployment}$$

Đây là giai đoạn đầu tiên và mang tính nền tảng nhất của quy trình CRISP-DM
— mọi quyết định kỹ thuật ở các giai đoạn sau (chọn thuật toán, ngưỡng lọc
outlier, cách diễn giải kết quả) đều phải truy ngược được về một mục tiêu
nghiệp vụ cụ thể được xác lập tại đây. Một sai lầm phổ biến của các đồ án
khai phá dữ liệu là bỏ qua hoặc làm sơ sài giai đoạn này, dẫn tới tình
trạng "có mô hình nhưng không biết áp dụng cho ai, để làm gì" — notebook
này cố gắng tránh sai lầm đó bằng cách trình bày đầy đủ các thành phần
chuẩn của Business Understanding theo tài liệu tham chiếu CRISP-DM 1.0.

## 1.1. Bối cảnh & Động lực (Business Background)

### 1.1.1. Bối cảnh ngành

Ngành taxi truyền thống tại New York City đang trải qua giai đoạn cạnh
tranh khốc liệt nhất trong lịch sử của nó. Kể từ khi các nền tảng gọi xe
công nghệ (Uber, Lyft) gia nhập thị trường, số lượng chuyến đi bằng Yellow
Taxi đã sụt giảm đáng kể qua từng năm, trong khi chi phí vận hành (giấy
phép medallion, bảo hiểm, nhiên liệu) gần như không đổi. Trong bối cảnh
đó, việc tối ưu hoá từng khía cạnh vận hành — từ thu nhập của tài xế đến
hiệu quả điều phối xe — trở thành yếu tố sống còn đối với các doanh nghiệp
vận hành taxi truyền thống.

### 1.1.2. Nguồn dữ liệu sẵn có

New York City Taxi & Limousine Commission (TLC) — cơ quan quản lý taxi và
limousine của thành phố — công bố công khai dữ liệu chi tiết từng chuyến
đi của Yellow Taxi theo từng tháng, bao gồm: thời gian đón/trả khách, toạ
độ khu vực, quãng đường, cước phí, hình thức thanh toán và tiền tip. Đây
là một trong số ít bộ dữ liệu giao thông đô thị quy mô lớn (hàng triệu bản
ghi/tháng), có độ chi tiết cao, và được công khai miễn phí — tạo điều kiện
lý tưởng để áp dụng các kỹ thuật khai phá dữ liệu nhằm trích xuất tri thức
phục vụ ra quyết định.

### 1.1.3. Vấn đề nghiệp vụ cụ thể

Từ bối cảnh trên, hai vấn đề nghiệp vụ cụ thể, có giá trị thực tiễn cao và
**có thể giải quyết bằng khai phá dữ liệu** được xác định:

1. **Tài xế không có cơ sở dự đoán khả năng nhận tip cao.** Trong mô hình
   thu nhập của tài xế taxi tại Mỹ, tiền tip chiếm một tỷ trọng đáng kể
   trong tổng thu nhập, nhưng hiện tại tài xế hoàn toàn dựa vào kinh
   nghiệm cá nhân và cảm tính để quyết định nên hoạt động ở khu vực/khung
   giờ nào — không có công cụ định lượng hỗ trợ.
2. **Đơn vị điều phối không có công cụ định lượng để nhận diện mẫu hình
   nhu cầu.** Việc phân bổ xe hiện tại phần lớn dựa trên kinh nghiệm quản
   lý, dẫn đến tình trạng mất cân đối cung-cầu cục bộ: có khu vực/khung giờ
   thừa xe (tài xế chờ khách lâu, lãng phí thời gian) trong khi khu vực
   khác lại thiếu xe (khách phải chờ lâu, trải nghiệm dịch vụ kém).

## 1.2. Phân tích các bên liên quan (Stakeholder Analysis)

Việc xác định rõ ai sẽ sử dụng kết quả của dự án, và họ cần gì, là bước
thường bị bỏ qua nhưng có ảnh hưởng trực tiếp tới cách thiết kế bài toán
mining (đặc biệt là cách diễn giải và trình bày kết quả ở giai đoạn
Deployment).

| Bên liên quan | Vai trò | Lợi ích kỳ vọng từ dự án | Mức độ ảnh hưởng |
|---|---|---|---|
| Tài xế taxi | Người trực tiếp vận hành | Gợi ý khu vực/khung giờ có khả năng tip cao, tăng thu nhập | Cao |
| Đơn vị điều phối (dispatcher) | Quản lý phân bổ phương tiện | Bản đồ/phân cụm nhu cầu theo khu vực - khung giờ để điều phối chủ động | Cao |
| Quản lý doanh nghiệp taxi | Ra quyết định chiến lược | Dữ liệu định lượng hỗ trợ quyết định đầu tư, mở rộng đội xe | Trung bình |
| Hành khách | Người sử dụng dịch vụ | Gián tiếp hưởng lợi từ thời gian chờ xe giảm (nếu điều phối hiệu quả hơn) | Thấp (gián tiếp) |
| Nhóm thực hiện đồ án | Đơn vị phân tích | Ứng dụng kiến thức CRISP-DM vào bài toán thực tế, làm portfolio | — |

Trong hai nhóm đối tượng có mức ảnh hưởng cao (tài xế và đơn vị điều
phối), dự án ưu tiên thiết kế bài toán mining sao cho kết quả **có thể
hành động được** (actionable) đối với cả hai — đây là lý do vì sao cả DM1
lẫn DM2 đều được thiết kế để tạo ra khuyến nghị cụ thể, không dừng ở việc
báo cáo số liệu thống kê mô tả.

## 1.3. Mục tiêu nghiệp vụ (Business Objectives)

| Mã | Mục tiêu | Mô tả chi tiết | Chỉ số đo lường thành công (định tính) |
|---|---|---|---|
| **BO1** | Hỗ trợ tài xế tối ưu thu nhập | Cung cấp thông tin định lượng giúp tài xế nhận diện đặc điểm chuyến đi (khu vực, khung giờ, quãng đường) có khả năng tip cao, từ đó điều chỉnh chiến lược hoạt động | Tài xế có thể đưa ra quyết định "nên hoạt động ở đâu, khi nào" dựa trên dữ liệu thay vì chỉ dựa vào cảm tính |
| **BO2** | Hỗ trợ điều phối xe theo nhu cầu | Phát hiện các nhóm chuyến đi có mẫu hình tương đồng (khu vực, khung giờ, quãng đường) để làm cơ sở dự báo và phân bổ phương tiện hợp lý hơn, giảm tình trạng mất cân đối cung-cầu cục bộ | Đơn vị điều phối có bản đồ mẫu hình nhu cầu cụ thể theo khu vực - khung giờ, thay vì chỉ dựa vào kinh nghiệm quản lý |

### Câu hỏi nghiên cứu (Business Questions)

Từ hai mục tiêu trên, các câu hỏi nghiên cứu cụ thể được đặt ra:

**Nhóm câu hỏi phục vụ BO1:**
- Đặc điểm nào của một chuyến đi (thời gian, quãng đường, khu vực, hình
  thức thanh toán) có liên quan đến khả năng khách hàng tip ở mức cao?
- Khung giờ và khu vực nào có tỷ lệ chuyến tip cao vượt trội so với mức
  trung bình chung?

**Nhóm câu hỏi phục vụ BO2:**
- Các chuyến đi trong thành phố có thể được nhóm thành bao nhiêu loại nhu
  cầu di chuyển khác nhau (VD: chuyến đi sân bay dài, chuyến đi ngắn giờ
  cao điểm nội đô, chuyến đi đêm khuya)?
- Mỗi nhóm nhu cầu có đặc trưng gì về không gian (khu vực đón/trả) và thời
  gian (khung giờ, ngày trong tuần)?
- Có tồn tại mẫu hình bất đối xứng theo hướng di chuyển (VD: buổi sáng
  dòng người đổ về trung tâm, buổi chiều ngược lại) hay không, và mẫu hình
  đó có ý nghĩa gì cho việc điều phối xe theo hướng?

## 1.4. Tiêu chí thành công về nghiệp vụ (Business Success Criteria)

Tiêu chí thành công về nghiệp vụ **khác** với tiêu chí thành công của mô
hình khai phá dữ liệu (sẽ trình bày ở mục 1.7) — tiêu chí nghiệp vụ đánh
giá liệu dự án có thực sự tạo ra giá trị sử dụng được, không chỉ đánh giá
độ chính xác kỹ thuật của mô hình:

- Xây dựng được mô hình dự đoán khả năng tip cao với độ chính xác **đủ tin
  cậy để có ý nghĩa tham khảo thực tế** — không yêu cầu độ chính xác tuyệt
  đối, vì hành vi tip vốn chịu ảnh hưởng bởi nhiều yếu tố cá nhân/ngẫu
  nhiên không thể quan sát được từ dữ liệu giao dịch (tâm trạng khách hàng,
  thái độ phục vụ của tài xế...).
- Xác định được các cụm chuyến đi có **ranh giới rõ ràng và có thể diễn
  giải bằng ngôn ngữ nghiệp vụ** (đặt tên được cho từng cụm, VD: "cụm
  chuyến sân bay", "cụm giờ cao điểm nội đô") — một mô hình phân cụm dù có
  chỉ số thống kê tốt nhưng không diễn giải được sẽ không có giá trị sử
  dụng thực tế đối với đơn vị điều phối.
- Kết quả của cả hai bài toán phải **chuyển hoá được thành khuyến nghị
  hành động cụ thể** (actionable insight) — không chỉ dừng ở việc báo cáo
  con số thống kê mô tả đơn thuần.
- Toàn bộ giới hạn của dữ liệu và mô hình (VD: DM1 chỉ đại diện cho khách
  thanh toán thẻ) phải được **nêu rõ minh bạch**, không phóng đại khả năng
  áp dụng thực tế của kết quả.

## 1.5. Đánh giá tình hình hiện tại (Situation Assessment)

### 1.5.1. Kiểm kê nguồn lực (Inventory of Resources)

| Loại nguồn lực | Chi tiết |
|---|---|
| Dữ liệu | File Parquet công khai từ trang chính thức TLC (Yellow Taxi Trip Records) và bảng tra cứu khu vực (`taxi_zone_lookup.csv`), miễn phí, không yêu cầu xin phép sử dụng |
| Nhân lực | Nhóm 2 thành viên, phân công theo 2 bài toán mining (Classification / Clustering) |
| Công cụ | Python (pandas, numpy, scikit-learn, seaborn, matplotlib, scipy), Google Colab |
| Thời gian | Giới hạn trong khuôn khổ môn học (theo lịch trình cụ thể ở mục 1.9) |
| Phần cứng | Máy tính cá nhân / Colab — do giới hạn tài nguyên, dự án chỉ sử dụng dữ liệu 1 tháng thay vì toàn bộ lịch sử nhiều năm |

### 1.5.2. Yêu cầu, giả định và ràng buộc

**Giả định (Assumptions):**
- Dữ liệu TLC phản ánh trung thực các chuyến đi thực tế, đã qua bước kiểm
  duyệt của cơ quan quản lý trước khi công bố.
- Hành vi di chuyển và tip trong 1 tháng dữ liệu được sử dụng có thể đại
  diện tương đối cho mẫu hình chung (dù không phản ánh được yếu tố mùa vụ).

**Ràng buộc (Constraints):**
- Dữ liệu chỉ giới hạn ở Yellow Taxi (không bao gồm Uber/Lyft/Green Taxi),
  nên kết quả không đại diện cho toàn bộ thị trường vận tải hành khách NYC.
- Ràng buộc thời gian: đồ án thực hiện trong khuôn khổ môn học, không
  triển khai thực tế (real-world deployment) lên hệ thống vận hành thật.
- Ràng buộc phần cứng: giới hạn dữ liệu sử dụng ở quy mô 1 tháng.

### 1.5.3. Sổ đăng ký rủi ro (Risk Register)

| # | Rủi ro | Khả năng xảy ra | Mức độ ảnh hưởng | Biện pháp giảm thiểu |
|---|---|---|---|---|
| R1 | Dữ liệu tip bị lệch hệ thống do giao dịch tiền mặt không ghi nhận tip | Chắc chắn xảy ra (đã biết trước từ tài liệu TLC) | Cao — làm sai lệch hoàn toàn nhãn `high_tip` nếu không xử lý | Chỉ sử dụng giao dịch thanh toán thẻ cho bài toán DM1 |
| R2 | Dữ liệu khu vực (LocationID) là vùng (zone), không phải toạ độ chính xác | Chắc chắn (giới hạn cố hữu của dữ liệu TLC) | Trung bình — giới hạn độ chi tiết phân tích không gian | Chấp nhận giới hạn, nêu rõ trong báo cáo, không suy diễn ở độ chi tiết cao hơn dữ liệu cho phép |
| R3 | Tồn tại bản ghi lỗi/outlier (fare âm, trip_distance = 0, lỗi thiết bị) | Cao (phổ biến với dữ liệu giao dịch quy mô lớn) | Trung bình — có thể làm lệch kết quả mô hình nếu không xử lý | Kiểm tra kỹ ở Data Understanding, làm sạch có căn cứ ở Data Preparation |
| R4 | Một vendor cụ thể có thể có lỗi feed dữ liệu (null hệ thống) | Chưa xác định tại thời điểm lập kế hoạch, cần kiểm chứng ở Data Understanding | Trung bình | Kiểm định thống kê (Chi-square) trước khi quyết định xử lý null |
| R5 | Dữ liệu chỉ 1 tháng, không đại diện đủ cho biến động theo mùa | Chắc chắn (ràng buộc tài nguyên đã biết trước) | Thấp — trong phạm vi đồ án học thuật | Nêu rõ giới hạn trong Deployment, không suy rộng kết luận sang các tháng khác |

### 1.5.4. Thuật ngữ (Terminology / Glossary)

| Thuật ngữ | Định nghĩa |
|---|---|
| Trip | Một chuyến đi hoàn chỉnh từ điểm đón đến điểm trả khách |
| Tip percentage | Tỷ lệ `tip_amount / fare_amount`, dùng làm cơ sở gán nhãn "tip cao" hay "tip thấp" |
| High tip | Nhãn nhị phân, bằng 1 nếu `tip_percentage ≥ 15%` |
| PULocationID / DOLocationID | Mã định danh khu vực đón (Pickup) / trả (Dropoff) khách, tra cứu qua bảng zone lookup do TLC cung cấp |
| VendorID | Mã định danh hệ thống taximeter/nhà cung cấp dữ liệu (Creative Mobile Technologies, Curb Mobility, Myle Technologies, Helix) |
| RatecodeID | Mã loại cước áp dụng cho chuyến đi (tiêu chuẩn, sân bay, cước thoả thuận...) |
| Commuting asymmetry | Hiện tượng bất đối xứng giữa số lượt đến và rời khỏi một khu vực theo từng khung giờ, phản ánh mẫu hình đi lại có mục đích (đi làm/về nhà) |

## 1.6. Mục tiêu khai phá dữ liệu (Data Mining Goals)

Mục tiêu nghiệp vụ (Business Objectives) được ánh xạ thành các mục tiêu kỹ
thuật cụ thể — đây chính là "bản dịch" từ ngôn ngữ nghiệp vụ sang ngôn ngữ
khai phá dữ liệu, và là cầu nối quan trọng nhất giữa Business Understanding
và các giai đoạn kỹ thuật tiếp theo.

| Mã | Bài toán | Loại học máy | Kỹ thuật dự kiến | Biến mục tiêu | Ánh xạ tới |
|---|---|---|---|---|---|
| **DM1** | Phân loại chuyến đi có khả năng tip cao hay không | Có giám sát (Supervised) | Classification (Logistic Regression / Random Forest / XGBoost) | `high_tip` (nhị phân) | BO1 |
| **DM2** | Phân cụm các chuyến đi theo đặc trưng thời gian - không gian - quãng đường | Không giám sát (Unsupervised) | Clustering (K-Means, đối chiếu Elbow Method + Silhouette Score) | Không có — mục tiêu là khám phá cấu trúc | BO2 |

**Vì sao chọn Classification cho DM1, không phải Regression?** Mục tiêu
nghiệp vụ (BO1) là giúp tài xế đưa ra quyết định hành động nhị phân ("có
nên ưu tiên khu vực này không") hơn là cần biết chính xác số tiền tip dự
kiến — một dự đoán nhị phân "khả năng tip cao hay không" dễ diễn giải và
hành động hơn một con số hồi quy chính xác nhưng khó chuyển hoá thành
quyết định thực tế.

**Vì sao chọn Clustering (không giám sát) cho DM2, không phải phân loại
khu vực có sẵn?** Vì mục tiêu là **khám phá** mẫu hình nhu cầu tiềm ẩn mà
chưa ai định nghĩa trước (không có nhãn "loại nhu cầu" có sẵn trong dữ
liệu) — đây đúng là bản chất của bài toán không giám sát, phù hợp hơn việc
áp đặt các nhãn phân loại chủ quan từ trước.

## 1.7. Tiêu chí thành công của khai phá dữ liệu (Data Mining Success Criteria)

Khác với tiêu chí nghiệp vụ (mục 1.4, mang tính định tính), tiêu chí ở đây
được lượng hoá bằng các độ đo thống kê cụ thể — đóng vai trò là "bài kiểm
tra" khách quan ở giai đoạn Evaluation.

### DM1 (Classification)

$$
\text{Precision} = \frac{TP}{TP + FP}, \qquad
\text{Recall} = \frac{TP}{TP + FN}, \qquad
F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

- Mô hình đạt **AUC-ROC hoặc F1-score vượt rõ rệt so với baseline** (mô
  hình dự đoán ngẫu nhiên hoặc luôn chọn lớp đa số) — baseline cụ thể sẽ
  được tính toán từ tỷ lệ phân bố thực tế của `high_tip` ở bước Data
  Preparation.
- Mô hình cho phép **diễn giải được đặc trưng quan trọng nhất** (feature
  importance) — vì mục tiêu nghiệp vụ (BO1) không chỉ cần một dự đoán
  chính xác mà cần *hiểu được lý do*, để chuyển hoá thành khuyến nghị cho
  tài xế.
- Do đã biết trước khả năng mất cân bằng lớp (`high_tip`), **không dùng
  Accuracy làm độ đo chính** — Accuracy có thể gây hiểu lầm nghiêm trọng
  với dữ liệu mất cân bằng (VD: một mô hình luôn dự đoán "không tip cao"
  vẫn có thể đạt Accuracy cao giả tạo nếu lớp đa số chiếm ưu thế).

### DM2 (Clustering)

$$
\text{Silhouette}(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}
$$

với $a(i)$ là khoảng cách trung bình từ điểm $i$ tới các điểm khác trong
cùng cụm, $b(i)$ là khoảng cách trung bình nhỏ nhất tới các điểm ở cụm
khác.

- Chỉ số **Silhouette Score đạt mức chấp nhận được** (thường > 0.3 được
  xem là có cấu trúc cụm hợp lý trong tài liệu tham khảo phổ biến).
- Số cụm $k$ được chọn thông qua **phương pháp định lượng** (Elbow Method
  dựa trên within-cluster sum of squares — WCSS), không chọn cảm tính.
- Mỗi cụm phải **đặt tên và diễn giải được bằng ngôn ngữ nghiệp vụ** dựa
  trên đặc trưng trung bình của cụm đó (VD: cụm có `trip_distance` và
  `trip_duration_min` cao vượt trội, `dropoff_borough` tập trung ở khu vực
  sân bay → đặt tên "cụm chuyến sân bay").

## 1.8. Kế hoạch dự án (Project Plan)

| Giai đoạn CRISP-DM | Công việc chính | Người phụ trách | Trạng thái |
|---|---|---|---|
| Business Understanding | Xác định mục tiêu, câu hỏi nghiên cứu, kế hoạch dự án (notebook này) | Cả nhóm | Hoàn thành |
| Data Understanding | Đọc dữ liệu, EDA, kiểm tra chất lượng dữ liệu, kiểm định thống kê | Cả nhóm (chủ trì luân phiên) | Hoàn thành |
| Data Preparation | Làm sạch dữ liệu, feature engineering, tích hợp, xuất dữ liệu cho DM1/DM2 | Cả nhóm | Hoàn thành |
| Modeling — DM1 (Classification) | Train/Test split, huấn luyện mô hình, so sánh thuật toán | Thành viên 1 | Chưa thực hiện |
| Modeling — DM2 (Clustering) | Chuẩn hoá dữ liệu, huấn luyện K-Means, chọn số cụm | Thành viên 2 | Chưa thực hiện |
| Evaluation | Đánh giá theo tiêu chí mục 1.7, đối chiếu với mục tiêu nghiệp vụ | Cả nhóm | Chưa thực hiện |
| Deployment | Tổng hợp báo cáo, khuyến nghị hành động cụ thể | Cả nhóm | Chưa thực hiện |

**Nguyên tắc phân công:** vì hai bài toán DM1 và DM2 độc lập nhau về mặt
kỹ thuật (dùng 2 tập dữ liệu riêng, đã tách sẵn ở Data Preparation), mỗi
thành viên có thể làm việc song song trên notebook Modeling của bài toán
mình phụ trách mà không phụ thuộc lẫn nhau — giảm thiểu xung đột khi làm
việc nhóm.

## 1.9. Tóm tắt & Chuyển giao sang giai đoạn tiếp theo

Notebook này đã thiết lập đầy đủ nền tảng cho toàn bộ dự án:

- **Hai mục tiêu nghiệp vụ cụ thể** (BO1, BO2), gắn với bối cảnh thực tế
  của ngành taxi NYC và có bên liên quan rõ ràng (tài xế, đơn vị điều
  phối).
- **Hai bài toán mining tương ứng** (DM1 Classification, DM2 Clustering),
  với tiêu chí thành công được lượng hoá cụ thể bằng công thức thống kê.
- **Các rủi ro đã được nhận diện trước** (đặc biệt là vấn đề tip tiền mặt
  — R1), giúp các quyết định xử lý dữ liệu ở giai đoạn sau (VD: lọc
  `payment_type = 1` cho DM1) có căn cứ rõ ràng ngay từ đầu, không phải xử
  lý phát sinh tuỳ tiện.
- **Kế hoạch phân công cụ thể** cho nhóm 2 người, tận dụng tính độc lập
  giữa hai bài toán mining.

Toàn bộ nội dung ở đây là **điểm tham chiếu** cho mọi quyết định kỹ thuật
ở các notebook tiếp theo — khi cần biện minh cho một lựa chọn xử lý dữ
liệu hay mô hình, nên quay lại đối chiếu với mục tiêu và tiêu chí đã đặt
ra tại đây. Bước tiếp theo: **Data Understanding**, khám phá và kiểm chứng
chất lượng dữ liệu thực tế trước khi tiến hành xử lý.